# Warm-start from Best Adapter + Fine-tune `up_proj` & `down_proj` (shared MoE layers)

**Strategy:**
1. Load Nemotron-3-Nano-30B base model
2. Load best existing adapter (0.86) as warm-start weights into 9 original modules
3. Freeze all LoRA params except `up_proj` and `down_proj` (which cover shared_experts linear layers)
4. Fine-tune only those layers on corrected v3 data
5. Save checkpoint every 10 steps + eval loss every 10 steps to find best checkpoint

> Note: `shared_experts` container cannot be targeted directly by PEFT.
> Its internal linear layers (`up_proj`, `down_proj`) are already in our module list.

In [ ]:

import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_MODEL_NAME      = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
EXISTING_ADAPTER_DIR = "/kaggle/input/datasets/zuhairsan/submission-tinker086"
DATASET_PATH         = "/kaggle/input/datasets/zuhairsan/new-cot-085/problem_ids_matched_v4.csv"
ADAPTER_DIR          = "/kaggle/working/sft_adapter"
OUTPUT_DIR           = "/kaggle/working"

# ── Sample budget (None = use all) ─────────────────────────────────────────
TYPE_SAMPLES = {
    "bit_manipulation":        None,
    "cipher":                  None,
    "cryptarithm_deduce":      None,
    "cryptarithm_guess":       None,
    "equation_numeric_deduce": None,
    "equation_numeric_guess":  None,
    "unit_conversion":         None,
    "numeral":                 None,
    "gravity":                 None,
}

# ── Model ──────────────────────────────────────────────────────────────────
MAX_SEQ_LEN = 8192

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_RANK    = 32
LORA_ALPHA   = 32
LORA_DROPOUT = 0.0

ALL_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "in_proj", "out_proj", "up_proj", "down_proj",
    "lm_head",
]
TRAINABLE_MODULES = ["shared_experts"]   # freeze logic uses 'shared_experts' in name

# ── Training ───────────────────────────────────────────────────────────────
SEED                   = 123
NUM_EPOCHS             = 1
BATCH_SIZE             = 2
GRAD_ACCUM             = 16     # eff_batch = 32
LR                     = 5e-5
LR_SCHEDULER           = "cosine"
WARMUP_STEPS           = 5
WEIGHT_DECAY           = 0.0
MAX_GRAD_NORM          = 1e9
ADAM_BETA1             = 0.9
ADAM_BETA2             = 0.95
ADAM_EPSILON           = 1e-8
EVAL_STEPS             = 25    # eval loss every 25 steps (monitoring only, no disk saves)
DATALOADER_NUM_WORKERS = 2

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

print("Config ready.")
print(f"  existing adapter : {EXISTING_ADAPTER_DIR}")
print(f"  dataset          : {DATASET_PATH}")
print(f"  trainable only   : shared_experts")
print(f"  microbatch       : {BATCH_SIZE}  grad_accum={GRAD_ACCUM}  eff_batch={BATCH_SIZE*GRAD_ACCUM}")
print(f"  lr={LR}, scheduler={LR_SCHEDULER}, eval every {EVAL_STEPS} steps (no checkpoint saves)")


## Setup & Model Loading

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found — check dataset input.")

whl = candidates[0]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", whl], check=True)
print(f"Installed: {whl}")

In [ ]:
import sys, os, shutil, stat

sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
ptxas_dst = '/tmp/ptxas-blackwell'
if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
    shutil.copy2(ptxas_src, ptxas_dst)
    os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

import triton.backends.nvidia.compiler as nv_compiler
nv_compiler.get_ptxas_version = lambda arch: '12.0'
print('Environment fixes applied.')

In [ ]:
import glob, os, subprocess, sys

packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
all_mamba  = sorted(glob.glob("/kaggle/input/**/*mamba_ssm*.whl", recursive=True))
all_causal = sorted(glob.glob("/kaggle/input/**/*causal*conv1d*.whl", recursive=True))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--no-index", "--find-links", packages_dir,
     "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes"],
    check=True,
)
if all_causal:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", all_causal[-1]], check=True)
if all_mamba:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", all_mamba[-1]], check=True)
print("Packages installed.")

In [ ]:
import torch
import kagglehub
from unsloth import FastLanguageModel
from peft import set_peft_model_state_dict, load_peft_weights

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
print(f"Model path: {MODEL_PATH}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    trust_remote_code=True,
    unsloth_force_compile=False,
    attn_implementation="eager",
    dtype=torch.bfloat16,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Base model loaded.")

In [ ]:

from unsloth import FastLanguageModel
from peft import set_peft_model_state_dict, load_peft_weights

# Create LoRA with the same 9 modules as the 0.86 adapter
print("Wrapping model with LoRA...")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=ALL_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()

# Load 0.86 adapter weights as warm-start
# Fix key naming: old Unsloth used "model.model.layers", new uses "model.backbone.layers"
print(f"\nLoading warm-start weights from: {EXISTING_ADAPTER_DIR}")
weights_raw = load_peft_weights(EXISTING_ADAPTER_DIR)

# Remap keys if needed
def remap_keys(weights):
    remapped = {}
    n_remapped = 0
    for k, v in weights.items():
        new_k = k.replace("base_model.model.model.", "base_model.model.backbone.")
        if new_k != k:
            n_remapped += 1
        remapped[new_k] = v
    print(f"  Remapped {n_remapped}/{len(weights)} keys (model.model → model.backbone)")
    return remapped

weights = remap_keys(weights_raw)
result = set_peft_model_state_dict(model, weights, adapter_name="default")
n_loaded = len(weights) - len(result.unexpected_keys)
print(f"  Loaded: {n_loaded}/{len(weights)} weights")
if result.unexpected_keys:
    print(f"  Still unexpected (first 3): {result.unexpected_keys[:3]}")
if result.missing_keys:
    se_missing = [k for k in result.missing_keys if 'shared_experts' in k]
    other_missing = [k for k in result.missing_keys if 'shared_experts' not in k]
    print(f"  shared_experts missing (random init, expected): {len(se_missing)}")
    if other_missing:
        print(f"  WARNING — other missing keys: {other_missing[:3]}")
print("Warm-start complete.")

# Freeze all LoRA params EXCEPT shared_experts (match full parameter path)
print("\nFreezing all LoRA params except shared_experts...")
frozen, trainable = 0, 0
trainable_names = []
for name, param in model.named_parameters():
    if param.requires_grad:
        if 'shared_experts' in name:
            trainable += param.numel()
            trainable_names.append(name)
        else:
            param.requires_grad = False
            frozen += param.numel()

print(f"  Frozen   : {frozen:,} params")
print(f"  Trainable: {trainable:,} params (shared_experts only)")
print(f"  Sample trainable names: {trainable_names[:3]}")
assert trainable > 0, "BUG: no trainable parameters — shared_experts LoRA not found!"


## Training

In [ ]:

import pandas as pd
import random, re, time, math, gc
from collections import defaultdict
from datasets import Dataset as HFDataset
from trl import SFTTrainer, SFTConfig
from torch.utils.data import DataLoader, Sampler

df = pd.read_csv(DATASET_PATH)
print(f"Loaded: {len(df):,} rows")

sampled_dfs = []
for type_name, n in TYPE_SAMPLES.items():
    subset = df[df["type"] == type_name]
    if len(subset) == 0:
        print(f"  WARNING: no rows for '{type_name}'")
        continue
    if n is None or n >= len(subset):
        sampled_dfs.append(subset)
        print(f"  {type_name:<28} {len(subset):>5} / {len(subset)} (all)")
    else:
        sampled_dfs.append(subset.sample(n, random_state=SEED))
        print(f"  {type_name:<28} {n:>5} / {len(subset)}")

df_sampled = pd.concat(sampled_dfs).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"\nTotal training rows: {len(df_sampled):,}")

records, record_types = [], []
for _, row in df_sampled.iterrows():
    prompt = str(row["prompt"])
    answer = str(row["answer"])
    cot    = str(row["generated_cot"])
    if not cot or cot == "nan" or len(cot.strip()) < 5:
        continue
    cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
    records.append({"messages": [
        {"role": "user",      "content": prompt + PROMPT_SUFFIX},
        {"role": "assistant", "content": cot_cleaned + f"\n</think>\n\\boxed{{{answer}}}"},
    ]})
    record_types.append(str(row["type"]))

dataset = HFDataset.from_list(records)
print(f"SFT records: {len(records):,}")

def formatting_prompts_func(example):
    messages = example["messages"]
    if messages and isinstance(messages[0], dict):
        conversations = [messages]
    else:
        conversations = messages
    texts = []
    for conv in conversations:
        try:
            text = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False, enable_thinking=True)
        except TypeError:
            text = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return texts

steps_per_epoch = max(1, len(dataset) // max(1, BATCH_SIZE * GRAD_ACCUM))
total_steps = steps_per_epoch * NUM_EPOCHS
print(f"Total steps: ~{total_steps}")

training_args = SFTConfig(
    output_dir=ADAPTER_DIR,           # write directly to working dir, no tmp
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=WARMUP_STEPS,
    max_length=MAX_SEQ_LEN,
    adam_beta1=ADAM_BETA1,
    adam_beta2=ADAM_BETA2,
    adam_epsilon=ADAM_EPSILON,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=EVAL_STEPS,
    # ── No checkpoint saving — avoids disk space exhaustion ──────────────
    save_strategy="no",
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    load_best_model_at_end=False,
    # ─────────────────────────────────────────────────────────────────────
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    remove_unused_columns=False,
    seed=SEED,
    report_to="none",
    packing=False,
    optim="adamw_8bit",
)

# 5% validation set
val_size = max(10, int(len(records) * 0.05))
val_indices = set(random.Random(SEED).sample(range(len(records)), val_size))
train_indices = [i for i in range(len(records)) if i not in val_indices]
val_indices   = list(val_indices)
train_dataset = HFDataset.from_list([records[i] for i in train_indices])
val_dataset   = HFDataset.from_list([records[i] for i in val_indices])
print(f"Train: {len(train_dataset):,}  Val: {len(val_dataset):,}")

def build_stratified_index_order(labels, batch_size, seed):
    by_label = defaultdict(list)
    for idx, label in enumerate(labels):
        by_label[label].append(idx)
    rng = random.Random(seed)
    for idx_list in by_label.values():
        rng.shuffle(idx_list)
    n_batches = max(1, math.ceil(len(labels) / batch_size))
    batches = [[] for _ in range(n_batches)]
    batch_order = list(range(n_batches))
    rng.shuffle(batch_order)
    assigned = 0
    for label in sorted(by_label.keys()):
        for idx in by_label[label]:
            batches[batch_order[assigned % n_batches]].append(idx)
            assigned += 1
    return [idx for batch in batches for idx in batch]

class PrecomputedOrderSampler(Sampler):
    def __init__(self, order): self.order = list(order)
    def __iter__(self): return iter(self.order)
    def __len__(self): return len(self.order)

class StratifiedSFTTrainer(SFTTrainer):
    def __init__(self, *args, stratified_order=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.stratified_order = stratified_order

    def get_train_dataloader(self):
        if self.stratified_order is None:
            return super().get_train_dataloader()
        dataloader_kwargs = {
            "batch_size":         self.args.per_device_train_batch_size,
            "sampler":            PrecomputedOrderSampler(self.stratified_order),
            "collate_fn":         self.data_collator,
            "num_workers":        self.args.dataloader_num_workers,
            "pin_memory":         self.args.dataloader_pin_memory,
            "persistent_workers": self.args.dataloader_persistent_workers,
            "drop_last":          self.args.dataloader_drop_last,
        }
        if self.args.dataloader_num_workers > 0:
            dataloader_kwargs["prefetch_factor"] = self.args.dataloader_prefetch_factor
        return DataLoader(self.train_dataset, **dataloader_kwargs)

effective_batch = BATCH_SIZE * GRAD_ACCUM
train_types = [record_types[i] for i in train_indices]
stratified_order = build_stratified_index_order(train_types, effective_batch, SEED)
print(f"Effective batch: {effective_batch}, stratified order built.")

trainer = StratifiedSFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    formatting_func=formatting_prompts_func,
    stratified_order=stratified_order,
)

torch.cuda.reset_peak_memory_stats()
print(f"GPU before training: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

print("Starting training (shared_experts only, no checkpoint saves)...")
t0 = time.time()
train_out = trainer.train()
elapsed = time.time() - t0
print(f"Training done in {elapsed/60:.1f} min")
print(f"Peak GPU: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

# Save final adapter only
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Final adapter saved to {ADAPTER_DIR}")


## Create submission.zip

In [ ]:
import json, os, shutil, zipfile

SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]
for fname in required_files:
    src = os.path.join(ADAPTER_DIR, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path) as f:
    cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"]   = 0.0
with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

print(f"\nsubmission.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB")
print("Done!")